# Evaluation of extracted metadata
---

In this portion of the solution, we will fetch all the completions and extract the `JSON` schema from the user provided document in the `results` directory. This `JSON` and `CSV` file containing the schema and metrics (such as latency, prompt and completion tokens) can be reviewed by a human in the loop or an LLM as a judge before applying it to downstream applications.

In [1]:
import re
import os
import json
import yaml
import time
import boto3
import globals
import zipfile
import logging
import litellm
import datetime
import requests
from utils import *
from globals import *
from io import BytesIO
from pathlib import Path
from typing import Union, Dict, Optional
from botocore.exceptions import ClientError
from litellm import completion, RateLimitError
from litellm.llms.bedrock.common_utils import BedrockError

In [2]:
# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers
logger.handlers.clear()

# Add a simple handler
handler = logging.StreamHandler()
formatter = logging.Formatter('[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

## Load the config file

In [ ]:
# Get the absolute path to the config file
# This config file contains data about the directory paths, the API specs that
# are used to generate the code, and the agent foundation models that are used to generate the code.
config_data = load_config("config.yaml")

In [ ]:
results_dir = Path(config_data['dir_info']['results_dir'])
json_files = list(results_dir.glob('**/*.json'))
logger.info(f"Total number of metadata documents extracted: {len(json_files)}")

In [ ]:
import pandas as pd
# Load each JSON file into a list of records
records = []
for file_path in json_files:
    with file_path.open('r', encoding='utf-8') as f:
        record = json.load(f)
        records.append(record)

metadata_df = pd.DataFrame(records)
metadata_df.head(10)

In [ ]:
os.path.join(results_dir, config_data['dir_info']['schema_dir'])

In [7]:
def sanitize_filename(s: str) -> str:
    """
    Replace characters that are not allowed in file names with an underscore.
    This regex removes characters like \ / * ? : " < > | 
    """
    return re.sub(r'[\\/*?:"<>|]', "_", s)

metadata_df['file_name'] = metadata_df['file_name'].astype(str)
schema_fpath: str = os.path.join(results_dir, config_data['dir_info']['schema_dir'])
os.makedirs(schema_fpath, exist_ok=True)

In [ ]:
for idx, row in metadata_df.iterrows():
    file_name_val = row.get("file_name")
    if file_name_val == 'nan' or file_name_val == 'None':
        logger.warning(f"Skipping row {idx}: Invalid filename")
        continue

    # Sanitize the file name and model_id to remove invalid characters.
    base_filename = os.path.splitext(file_name_val)[0]
    clean_filename = sanitize_filename(base_filename)
    model_id_sanitized = sanitize_filename(row.get("model_id"))

    # Construct the output file path
    output_filename = os.path.join(
        schema_fpath,
        f"{model_id_sanitized}_{clean_filename}_completion.json"
    )

    # Load the JSON content and save to file
    try:
        completion_json = json.loads(row.get("completion"))
    except Exception as e:
        logger.error(f"Error loading JSON from row {idx}: {e}")
        continue
    logger.info(f"Processed and loaded the completion from file: {clean_filename}")
    try:
        with open(output_filename, 'w') as f:
            json.dump(completion_json, f, indent=2)
    except Exception as e:
        logger.error(f"Error writing file {output_filename}: {e}")

Next, view all metadata extracted in the [`results/schema_extracted`](results/schema_extracted) directory. This directory contains metadata extracted from each user provided document. View an example below:

``` {.json}
{
  "document_metadata": {
    "title": "Machine Learning Best Practices in Healthcare and Life Sciences",
    "author": "Sai Sharanya Nalla, Wajahat Aziz, Kartik Kannapur, Ujjwal Ratan, Garin Kessler, Ian Sutcliffe",
    "creation_date": "November 22, 2021",
    "last_modified": "November 22, 2021",
    "page_count": 42,
    "file_size": "no"
  },
  "content_analysis": {
    "executive_summary": "This whitepaper describes how AWS approaches machine learning (ML) in a regulated environment and provides guidance on good ML practices using AWS products. It takes into consideration the principles described in the FDA's Proposed Regulatory Framework for Modifications to AI/ML-Based Software as a Medical Device (SaMD) discussion paper.",
    "key_topics": [
      "Machine Learning in Healthcare",
      "GxP Compliance",
      "Model Training",
      "Model Validation",
      "Model Monitoring",
      "Data Security",
      "Model Interpretability",
      "MLOps",
      "AWS Services for ML"
    ],
    "language_detection": {
      "primary_language": "English",
      "secondary_languages": [
        "no"
      ]
    }
  },
  "structural_elements": {
    "sections": [
      {
        "heading": "Introduction",
        "content": "Overview of ML adoption in pharmaceutical industry and regulatory challenges",
        "page_number": 1
      },
      {
        "heading": "Benefits of machine learning",
        "content": "Discussion of ML benefits in healthcare and regulatory considerations",
        "page_number": 3
      },
      {
        "heading": "Reference architectures",
        "content": "Details of training pipeline and inference pipeline architectures",
        "page_number": 31
      }
    ],
    "tables": [
      {
        "table_id": "no",
        "caption": "no",
        "data": [],
        "page_location": "no"
      }
    ],
    "figures": [
      {
        "figure_id": "1",
        "caption": "Trade-off between performance and model interpretability",
        "image_type": "diagram",
        "page_location": 24
      },
      {
        "figure_id": "2",
        "caption": "MLOps workflow",
        "image_type": "diagram",
        "page_location": 29
      },
      {
        "figure_id": "3",
        "caption": "Model training and tracking architecture",
        "image_type": "diagram",
        "page_location": 31
      }
    ]
  },
  "reference_information": {
    "citations": [
      "no"
    ],
    "external_links": [
      "https://aws.amazon.com/health/customer-stories",
      "https://aws.amazon.com/health/lifesciences-partner-solutions",
      "https://aws.amazon.com/sagemaker/",
      "https://aws.amazon.com/glue/",
      "https://aws.amazon.com/codepipeline/"
    ],
    "footnotes": [
      "no"
    ]
  },
  "semantic_analysis": {
    "sentiment_score": "no",
    "key_entities": [
      {
        "entity": "AWS",
        "type": "organization",
        "mentions": 42
      },
      {
        "entity": "FDA",
        "type": "organization",
        "mentions": 8
      },
      {
        "entity": "Amazon SageMaker",
        "type": "product",
        "mentions": 15
      }
    ],
    "document_category": "Technical Whitepaper"
  }
}
```